## Installing Dependencies & Load Text and PDF,s

In [2]:
#!pip install langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [3]:
from langchain_core.documents import Document

In [4]:
sample_doc = Document(
    page_content = "Hello World!",
    metadata = {"source":"https://www.google.com"}
)

In [5]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [6]:
type(sample_doc)

langchain_core.documents.base.Document

In [7]:
# Text data
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
path = "/content/drive/MyDrive/RAG Data/python.txt"

from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("/content/drive/MyDrive/RAG Data/python.txt",encoding = "utf-8")
documents = loader.load()

print(documents)

[Document(metadata={'source': '/content/drive/MyDrive/RAG Data/python.txt'}, page_content='--------OPEN IN BROWSER-------------------\n\nhttps://lwfiles.mycourse.app/62a6cd5e1e9e2fbf212d608d-public/publicFiles/Python.txt')]


In [9]:
document = loader.load()

In [10]:
document

[Document(metadata={'source': '/content/drive/MyDrive/RAG Data/python.txt'}, page_content='--------OPEN IN BROWSER-------------------\n\nhttps://lwfiles.mycourse.app/62a6cd5e1e9e2fbf212d608d-public/publicFiles/Python.txt')]

In [11]:
import requests

url = "https://lwfiles.mycourse.app/62a6cd5e1e9e2fbf212d608d-public/publicFiles/Python.txt"

response = requests.get(url)

print(response.status_code)
print(response.text)

200
ï»¿Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.
Some key features and benefits of Python include:
* Ease of Learning: Simple syntax and readability make Python beginner-friendly.
* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and more.
* Extensive Libraries and Frameworks: Libraries like Num

In [12]:
# PDF Data Loading

# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("/content/drive/MyDrive/RAG Data/research2.pdf")

# document = pdf_loader.load()

# document

In [13]:
# PDF Data Loading

# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("/content/drive/MyDrive/RAG Data/research2.pdf")

# document = pdf_loader.load()

# document

## Ingestion Pipeline

In [14]:
# Data => Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

## Converting into Documents

In [15]:
def load_all_pdfs():
    folder_path = "/content/drive/MyDrive/RAG Data/"
    num_docs = 0
    all_docs = []
    
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path create
            pdf_path = os.path.join(folder_path,filename)
            
            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs += 1
    
    print("Total pdfs : ",num_docs)
    print("Total pages : ",len(all_docs))
    return all_docs

In [16]:
all_pdf_documents = load_all_pdfs()

Total pdfs :  2
Total pages :  32


In [17]:
type(all_pdf_documents[0])

langchain_core.documents.base.Document

## Spliting into Chunks

In [20]:
# Chunks
#!pip install langchain_text_splitters

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents,chunk_size = 500,chunk_overlap = 50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )
    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [22]:
chunks = split_docs(all_pdf_documents)

In [23]:
len(chunks)

320

In [24]:
#chunks

## Vector Embeding

In [25]:
from sentence_transformers import SentenceTransformer

In [30]:
class EmbedingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name.strip()

        print("Loading Model ...", self.model_name)

        self.model = SentenceTransformer(self.model_name)

        print(
            "Embedding Dimensions =",
            self.model.get_sentence_embedding_dimension()
        )
        
    def generate_embeding(self,text):
        embedings = self.model.encode(text,show_progress_bar = True)
        print("Embeding shape = ",embedings.shape)
        return embedings

In [31]:
embeding_manager = EmbedingManager()

Loading Model ... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding Dimensions = 384


## Vector Store

In [34]:
import chromadb
import uuid

In [40]:
class VectorStoreManager:
    def __init__(
        self,
        persist_directory="/content/drive/MyDrive/RAG Data/vector_store",
        collection_name="pdf_documents"
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        # create client
        self.client = chromadb.PersistentClient(
            path=self.persist_directory
        )

        # create collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "vector store collection for pdf embedings in RAG"
            }
        )

        print(
            "Intialize the vector store with collection",
            self.collection_name
        )
        print("docs in collection :", self.collection.count())

    def add_documents(self, documents, embedings):
        if len(documents) != len(embedings):
            raise ValueError(
                "Num of Documents does not match num of Embedings"
            )

        # store => ids, embeddings, documents, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embedings_list = []

        for i, (doc, embeding) in enumerate(zip(documents, embedings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embedings_list.append(embeding.tolist())

        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embedings_list
        )

        print("Total documents added in vector store =", len(documents_content))
        print("docs in collection =", self.collection.count())

In [41]:
vector_store = VectorStoreManager()

Intialize the vector store with collection pdf_documents
docs in collection : 0


In [46]:
# data => documents => chunks => embeding => store in vector store

texts = [doc.page_content for doc in chunks]

embeding = embeding_manager.generate_embeding(texts)

vector_store.add_documents(chunks,embeding)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Embeding shape =  (320, 384)
Total documents added in vector store = 320
docs in collection = 320


# Retrieval Pipeline

In [49]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class RAGRetriever:
    def __ inint__(self,embeding_manager,vector_store):
        self.embeding_manager = embeding_manager,
        self.vector_store = vector_store
        
    def retrieve(self,query,top_k =5,score_threshold =0.0):
        # Query => embeding
        query_embeding = self.embeding_manager.generate_embedings([query])[0]
        
        # sementic search
        results = self.vector_store.collection.query(
            query_embedings = [query_embedings.tolist()],
            n_results = top_k
        )
        
        # cosine similarity
        if results["documents"] and results["documenys"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]
            
            for i ,(doc_id,metadata,document,distance) in enumerate(zip(ids,metadatas,documents,distances)):
                similarity_score = 1 - distance
                
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id":doc_id,
                    })
            
        else:
            print("No documents found")
            
        return retrieved_docs